# TMDWF Adjacent-b Difference N-State Fit Template

This notebook runs the adjacent `(bT,bz)` difference workflow and writes reconstructed matrix-element tables compatible with the standard TMDWF downstream tools.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from lqcd_analysis.notebook_workflows import (
    pretty_print_config,
    render_tmdwf_nstate_diff_input_text,
    run_tmdwf_nstate_diff_from_notebook,
    validate_tmdwf_nstate_diff_notebook_config,
)


## User Inputs

The graph anchor is fixed at `bT=0, bz=0`; keep both zero entries in the requested lists.


In [ ]:
EXAMPLE_C2PT = REPO_ROOT / "examples" / "data" / "l64c64a076_m140" / "comb_c2pt_csv"
EXAMPLE_qTMDWF = REPO_ROOT / "examples" / "data" / "l64c64a076_m140" / "comb_qTMDWF"
EXAMPLE_2PT_RESULTS = REPO_ROOT / "examples" / "outputs" / "tmdwf_two_point_fit_ref"
EXAMPLE_OUTPUTS = REPO_ROOT / "examples" / "outputs" / "tmdwf_nstate_diff_fit_notebook"

workflow_config = {
    # Data settings
    "title_pattern": "l64c64a076_m140_fit_pz*",
    "ns": 64,
    "nt": 64,
    "lattice_spacing_fm": 0.076,
    "pzlist": [0],
    "gmlist": ["T5"],
    "etalist": ["eta0"],
    "Tdirlist": ["b_X", "b_Y"],
    "bTlist": [0, 1, 2],
    "bzlist": [0, 1, 2, 3],
    "qtmdwf_h5": str(EXAMPLE_qTMDWF / "qTMDWF_CG_1HYP_M140_GSRC_W52_k0_src5_O{gm}.h5"),
    "dataset_path_template": "SP/{gm}/PX0PY0PZ{pz}/{Tdir}/{eta}/bT{bT}/bz{bz}",
    "tsrange": [0, 20],

    # Two-point input
    "two_point_fit_root": str(EXAMPLE_2PT_RESULTS),
    "two_point_fit_window_by_pz": {0: [4, 12]},
    "c2pt": str(EXAMPLE_C2PT / "c2pt_5_5_k0_pz*_real.csv"),
    "fold_t": "periodic",

    # Difference-fit settings
    "fit_component": "both",
    "nstates": [1, 2],
    "fit_window": {0: [4, 12]},
    "binsize": 1,
    "bootstrap_samples": 32,
    "bootstrap_size": 32,
    "seed": 2026,
    "two_point_fit_sample_coupled": False,

    "results_dir": str(EXAMPLE_OUTPUTS),
}


## Option Guide

- `bTlist` and `bzlist` define graph nodes; adjacent sorted entries define edges.
- `(bT=0,bz=0)` is fit directly and used as the fixed local anchor.
- Edge fits use paired bootstrap differences `R_v(t)-R_u(t)` and a linear N-state design matrix.
- Main `fit` and `samples` outputs keep the standard TMDWF n-state naming so `tmdwf-normalize` and `tmdwf-fourier` can consume them as `input_root`.
- `diagnostics/` contains edge deltas, graph residuals, plaquette closure, and fixed-`bT` bz-chain comparisons.


In [ ]:
validated = validate_tmdwf_nstate_diff_notebook_config(workflow_config)
validated


In [ ]:
input_preview = render_tmdwf_nstate_diff_input_text(workflow_config)
print(input_preview)


In [ ]:
# outputs = run_tmdwf_nstate_diff_from_notebook(workflow_config)
# for output in outputs:
#     print(output)


In [ ]:
print(pretty_print_config(workflow_config))
